In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split

fmselect_load = np.load("../data/sprout/IBMMIX_o3mini.npy", allow_pickle=True)
data = fmselect_load.item()

prompts = data["prompts"]
categories = data["categories"]

models = list(data.keys())[2:]
print(models)
print(models[11])

In [2]:
cost_true = []
cost_pred = []
perf_true = []
perf_pred = []


for m in models:
    cost_true.append(data[m]['actual cost'])
    cost_pred.append(data[m]['predicted cost'])
    perf_true.append(data[m]['actual perf'])
    perf_pred.append(data[m]['predicted perf'])

cost_pred = np.array(cost_pred).T
cost_true = np.array(cost_true).T
perf_pred = np.array(perf_pred).T
perf_true = np.array(perf_true).T
perf_pred = 1/(1 + np.exp(-perf_pred))

infos = {}
for i, m in enumerate(models):
    infos[m] = dict(cost_pred = cost_pred[:, i].mean(), cost_true = cost_true[:, i].mean(),
                    perf_pred = perf_pred[:, i].mean(), perf_true = perf_true[:, i].mean())


perf_o3mini = perf_true[:, 11]
perf_others = np.delete(perf_true, 11, 1).sum(axis = 1)

def route(scores, cost, cost_pred, correctness, lamb_range = np.arange(0, 1.001, 0.001)):
    router_cost = np.zeros(shape = (scores.shape[0], lamb_range.shape[0]))
    router_perf = np.zeros_like(router_cost)
    
    model_idx_all = np.zeros_like(router_cost)
    
    for idx_lam, lam in enumerate(lamb_range):
        model_idx = ((1 - lam) * scores - lam * cost_pred * 1000).argmax(axis = 1, keepdims = True)
        router_perf[:, idx_lam] = np.take_along_axis(correctness, model_idx, axis = 1).reshape((-1))
        router_cost[:, idx_lam] = np.take_along_axis(cost, model_idx, axis = 1).reshape((-1))
        model_idx_all[:, idx_lam] = model_idx[:, 0]

    return router_cost, router_perf, model_idx_all

router_cost, router_perf, model_idx_all = route(perf_pred, cost_true, cost_pred, perf_true)


cost_mean = cost_true.mean(axis = 0)
perf_mean = perf_true.mean(axis = 0)
router_cost_mean = router_cost.mean(axis = 0)
router_perf_mean = router_perf.mean(axis = 0)


In [ ]:
import os
os.makedirs('../plots', exist_ok=True)

markers = ['o', 's', 'D', '^', 'v', 'p', '*', 'x', '+', 'h', 'H', 'd', '>']
colors = ['r', 'green','orange', 'k', 'b', 'y', 'purple']

data_names = ["gpqa", "MuSR", 'MMLU-Pro', 'MATH', 'openhermes', "ragbench"]

cost_ratios = []
perf_ratios = []

for i, cat in enumerate(data_names):
    idx = np.where(np.char.find(categories, cat)>=0)[0]

    cost_mean = cost_true[idx, :].mean(axis = 0)
    perf_mean = perf_true[idx, :].mean(axis = 0)
    router_cost_mean = router_cost[idx, :].mean(axis = 0)
    router_perf_mean = router_perf[idx, :].mean(axis = 0)

    gpt4o_cost = cost_mean[11]
    gpt4o_perf = perf_mean[11]

    cost_ratio = router_cost_mean/gpt4o_cost
    perf_ratio = router_perf_mean/gpt4o_perf

    cost_ratios.append(cost_ratio)
    perf_ratios.append(perf_ratio)

from math import pi

props = [0.1, 0.2, 0.3]
props_name = ["10% of GPT-4o cost",
              "20% of GPT-4o cost",
              "30% of GPT-4o cost"]
var = []
for p in props:
    var_d = []
    for d in range(len(data_names)):
        cost_d = cost_ratios[d]
        perf_d = perf_ratios[d]
        var_d.append(perf_d[cost_d <= p].max().round(4))
    var.append(var_d)

var = np.array(var)

df_dir = {'group': props_name}
for i, cat_name in enumerate(data_names):
    df_dir[cat_name] = var[:, i]

df = pd.DataFrame(df_dir)

categories_axes = list(df)[1:]
N = len(categories_axes)

angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig = plt.figure(figsize=(4, 4))
ax = fig.add_subplot(111, polar=True)
ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)
plt.xticks(angles[:-1], categories_axes)
ax.set_rlabel_position(0)

for i in range(len(props)):
    values = df.loc[i].drop('group').values.flatten().tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=1, linestyle='solid', label=props_name[i], color=colors[i])
    ax.fill(angles, values, color=colors[i], alpha=0.05)

N_0 = 1000
angles_0 = [n / float(N_0) * 2 * pi for n in range(N_0)]
ax.plot(angles_0, np.ones_like(angles_0), linewidth=1.5, linestyle='--', color="blue", label="GPT-4o")
ax.tick_params(axis='both', which='major', labelsize=12)

fig.legend(loc='upper right')
fig.savefig('../plots/sprout_spider.pdf', bbox_inches='tight')
plt.show()